In [1]:
from pymongo import MongoClient
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client["medical_data"]
collection = db["nhs_diseases"]

data = pd.DataFrame(list(collection.find()))

print("Dataset size:", len(data))

Dataset size: 348


In [2]:
def flatten_sections(sections):

    text = []

    for s in sections:
        title = s.get("title","")
        paragraphs = " ".join(s.get("paragraphs",[]))

        text.append(title + " " + paragraphs)

    return " ".join(text)

data["full_text"] = data["sections"].apply(flatten_sections)

In [3]:
import re

def clean_text(text):

    text = text.lower()
    text = re.sub(r'\s+',' ',text)

    return text.strip()

data["clean_text"] = data["full_text"].apply(clean_text)

In [4]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    tokens = word_tokenize(text)

    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and len(t) > 2
    ]

    return " ".join(tokens)

data["processed_text"] = data["clean_text"].apply(preprocess)

[nltk_data] Downloading package punkt to C:\Users\eng
[nltk_data]     Abdelrhman/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\eng
[nltk_data]     Abdelrhman/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\eng
[nltk_data]     Abdelrhman/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    data["processed_text"].tolist(),
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [6]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Total vectors indexed:", index.ntotal)

Total vectors indexed: 348


In [7]:
def search_disease(query, top_k=5):

    query_vec = model.encode([query]).astype("float32")

    D, I = index.search(query_vec, top_k)

    results = data.iloc[I[0]][["name"]]

    return results

In [8]:
result = search_disease("pain in heel while walking")

print(result)

                             name
233             Plantar heel pain
219  Pain in the ball of the foot
0           Achilles tendinopathy
136                   Indigestion
36         Bunion (hallux valgus)


In [9]:
def retrieve_candidates(query, k=10):

    query_vec = model.encode([query]).astype("float32")

    D, I = index.search(query_vec, k)

    candidates = data.iloc[I[0]]

    return candidates, query_vec

In [10]:
pairs = []
labels = []

for i,row in data.iterrows():

    disease_text = row["processed_text"]

    # positive pair
    pairs.append((disease_text, disease_text))
    labels.append(1)

    # negative pair
    neg = data.sample(1)["processed_text"].values[0]

    pairs.append((disease_text, neg))
    labels.append(0)

In [11]:
def pair_embedding(q,d):

    q_vec = model.encode(q)
    d_vec = model.encode(d)

    return np.concatenate([q_vec,d_vec])

In [12]:
import numpy as np

X = []
y = []

for (q,d),label in zip(pairs,labels):

    X.append(pair_embedding(q,d))
    y.append(label)

X = np.array(X)
y = np.array(y)

In [13]:
import tensorflow as tf

reranker = tf.keras.Sequential([

    tf.keras.layers.Dense(256,activation="relu",input_shape=(768,)),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(64,activation="relu"),

    tf.keras.layers.Dense(1,activation="sigmoid")

])

reranker.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

reranker.fit(
    X,
    y,
    epochs=10,
    batch_size=16
)

c:\Users\eng Abdelrhman\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
c:\Users\eng Abdelrhman\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4928 - loss: 0.6963
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5704 - loss: 0.6822
Epoch 3/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6149 - loss: 0.6578
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7802 - loss: 0.5726
Epoch 5/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8836 - loss: 0.3900
Epoch 6/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9583 - loss: 0.2069
Epoch 7/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9770 - loss: 0.1117
Epoch 8/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9943 - loss: 0.0499
Epoch 9/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9899 - loss: 0.0463
Epoch 10/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9943 - loss: 0.0378


In [14]:
def hybrid_search(query,k=5):

    candidates,query_vec = retrieve_candidates(query,10)

    scores = []

    for _,row in candidates.iterrows():

        disease_text = row["processed_text"]

        vec = pair_embedding(query,disease_text)

        score = reranker.predict(vec.reshape(1,-1))[0][0]

        scores.append(score)

    candidates["rerank_score"] = scores

    candidates = candidates.sort_values(
        "rerank_score",
        ascending=False
    )

    return candidates[["name","rerank_score"]].head(k)

In [15]:
result = hybrid_search("pain in heel while walking")

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
                             name  rerank_score
137               Ingrown toenail      0.327799
233             Plantar heel pain      0.323105
155                    Leg cramps      0.186642
219  Pain in the ball of the foot      0.171956
36         Bunion (hallux valgus)      0.161947


C:\Users\eng Abdelrhman\AppData\Local\Temp\ipykernel_8236\89268296.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates["rerank_score"] = scores


In [16]:
result = hybrid_search(
    "pain in the back of the heel when walking"
)

print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
                                                  name  rerank_score
317  Traction apophysitis of the hip in children an...      0.452573
242    Positional talipes in children and young people      0.252075
233                                  Plantar heel pain      0.205516
36                              Bunion (hallux valgus)      0.154141
136                                        Indigestion      0.149755


C:\Users\eng Abdelrhman\AppData\Local\Temp\ipykernel_8236\89268296.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates["rerank_score"] = scores
